# QTEMP objective + human temporal-disturbance review

This notebook forms the **union** of objective QTEMP-positive recordings and recordings rated as `Temporal discontinuities` by human QC (distributed main and crossed reliability sets). It preserves source/rater provenance, overlays objective and human intervals on waveform and spectrogram views, provides audio playback, and saves a separate manual verification table.

The objective QTEMP features are exploratory, not validated packet-loss labels. A visual or audible disturbance should be described phenomenologically; do not infer its network or physiological cause from this notebook. Run cells from top to bottom.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display, clear_output
import ipywidgets as widgets

def find_project_root(start=Path.cwd()):
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'config' / 'project.yaml').is_file():
            return candidate
    raise FileNotFoundError('Could not locate project root')

ROOT = find_project_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from paper1_qc.human_qc import load_interval_human_qc
from paper1_qc.media import decode_native_audio

DATA_ROOT = Path(r'C:\Users\musikicn\Desktop\Nevena_project\Data_13072026')
HUMAN_ROOT = DATA_ROOT / 'Bamboo_passage_HumanQC'
FREEZE = ROOT / 'MAIN outputs' / '00_DATA_FREEZE' / 'v1' / 'frozen_bamboo_recordings.csv'
QTEMP_ROOT = ROOT / 'MAIN outputs' / '02_FEATURE_REVIEWED' / '00_working_candidates' / 'temporal_discontinuity' / 'qtemp-v1.0.0-analytical-final-no-retained'
QTEMP_FEATURES = QTEMP_ROOT / 'tables' / 'qtemp_v100_exploratory_features.csv'
OBJECTIVE_EVENTS = ROOT / 'MAIN outputs' / '02_FEATURE_FREEZE' / 'temporal_discontinuity' / 'qtemp-v1.0.0-g9-deferred-override' / 'tables' / 'qtemp_v10_accepted_event_ledger.csv'
OUT = ROOT / 'outputs' / '06_qtemp_manual_review'
OUT.mkdir(parents=True, exist_ok=True)
RATERS = ['Abbas', 'Liya', 'Samaana', 'Samara']
print('Project:', ROOT)
print('Review outputs:', OUT)

In [ ]:
def read_human(source, design, exclusions=None):
    ratings, _, intervals, issues = load_interval_human_qc(
        source, rater_strategy='parent_directory', rater_directory_names=RATERS,
        exclude_path_parts=exclusions or [], interval_time_base='absolute'
    )
    ratings = ratings.loc[ratings['category'].eq('temporal_discontinuity')].copy()
    intervals = intervals.loc[intervals['family'].eq('temporal_discontinuity')].copy()
    ratings['design'] = design
    intervals['design'] = design
    return ratings, intervals, issues

main_ratings, main_intervals, main_issues = read_human(HUMAN_ROOT, 'distributed_main', ['Reliability'])
rel_ratings, rel_intervals, rel_issues = read_human(HUMAN_ROOT / 'Reliability', 'crossed_reliability')
human_ratings = pd.concat([main_ratings, rel_ratings], ignore_index=True)
human_intervals = pd.concat([main_intervals, rel_intervals], ignore_index=True)

# Reliability exports prepend a severity stratum before '__'; the suffix is the media filename.
for frame in (human_ratings, human_intervals):
    frame['media_file_name'] = frame['file_name'].astype(str).str.split('__').str[-1]
    frame['logical_recording_id'] = frame['media_file_name'].map(lambda x: Path(x).stem)
human_positive = human_ratings.loc[human_ratings['rating'].eq(1)].copy()
human_positive_intervals = human_intervals.merge(
    human_positive[['file_name', 'rater_id', 'design', 'logical_recording_id']].drop_duplicates(),
    on=['file_name', 'rater_id', 'design', 'logical_recording_id'], how='inner', validate='many_to_one'
)

metadata = pd.read_csv(FREEZE)
metadata = metadata[['logical_recording_id', 'SubjectID', 'selected_media_file_name', 'media_path']].copy()
metadata['media_key'] = metadata['selected_media_file_name'].astype(str).str.casefold()
features = pd.read_csv(QTEMP_FEATURES)
feature_cols = ['qtemp_dropout_duration_fraction', 'qtemp_dropout_event_rate_per_min',
                'qtemp_frozen_audio_duration_fraction', 'qtemp_frozen_audio_event_rate_per_min']
objective_positive = features.loc[(features[feature_cols].fillna(0) > 0).any(axis=1)].copy()
objective_events = pd.read_csv(OBJECTIVE_EVENTS)

human_files = pd.DataFrame({'human_annotation_file_name': sorted(human_positive['media_file_name'].unique())})
human_files['logical_recording_id'] = human_files['human_annotation_file_name'].map(lambda x: Path(x).stem)
human_match = human_files.merge(metadata, on='logical_recording_id', how='left')
# A few human exports reference MP4 while the freeze selected another extension, or lie outside the freeze.
# Resolve those by the exact recording stem without changing cohort membership.
media_index = {}
for path in (DATA_ROOT / 'Bamboo_passage_only').rglob('*'):
    if path.is_file() and path.suffix.lower() in {'.wav', '.webm', '.mp4'}:
        media_index.setdefault(path.stem.casefold(), path)
missing_path = human_match['media_path'].isna()
human_match.loc[missing_path, 'media_path'] = human_match.loc[missing_path, 'logical_recording_id'].map(
    lambda x: str(media_index.get(str(x).casefold(), ''))
)
human_match.loc[missing_path, 'selected_media_file_name'] = human_match.loc[missing_path, 'media_path'].map(
    lambda x: Path(x).name if x else ''
)
human_match.loc[missing_path, 'SubjectID'] = human_match.loc[missing_path, 'logical_recording_id'].str.split('_').str[0]
unmatched = human_match.loc[human_match['media_path'].astype(str).eq(''), 'human_annotation_file_name']
assert unmatched.empty, f'Human-QC audio files not found: {unmatched.tolist()}'
review_media = pd.concat([metadata.drop(columns='media_key'), human_match[metadata.columns.drop('media_key')]], ignore_index=True)
review_media['_path_available'] = review_media['media_path'].notna() & review_media['media_path'].astype(str).ne('')
review_media = review_media.sort_values('_path_available', ascending=False).drop_duplicates('logical_recording_id').drop(columns='_path_available')

candidates = pd.concat([
    objective_positive[['logical_recording_id']].assign(objective_positive=True),
    human_match[['logical_recording_id']].assign(human_positive=True),
], ignore_index=True).groupby('logical_recording_id', as_index=False).agg(
    objective_positive=('objective_positive', 'max'), human_positive=('human_positive', 'max')
).fillna(False)
candidates = candidates.merge(review_media, on='logical_recording_id', how='left', validate='one_to_one')
candidates = candidates.merge(features[['logical_recording_id', *feature_cols]], on='logical_recording_id', how='left', validate='one_to_one')
rating_summary = human_positive.groupby('logical_recording_id').agg(
    human_positive_ratings=('rating', 'sum'), human_raters=('rater_id', lambda x: ', '.join(sorted(set(x)))),
    human_designs=('design', lambda x: ', '.join(sorted(set(x))))
).reset_index()
candidates = candidates.merge(rating_summary, on='logical_recording_id', how='left')
candidates['source_group'] = np.select(
    [candidates['objective_positive'] & candidates['human_positive'], candidates['objective_positive']],
    ['objective + human', 'objective only'], default='human only'
)
candidates = candidates.sort_values(['source_group', 'selected_media_file_name']).reset_index(drop=True)

candidates.to_csv(OUT / 'temporal_review_recordings.csv', index=False)
human_positive[['design','rater_id','media_file_name','annotated_duration_sec','event_count_union','source_file']].to_csv(OUT / 'human_temporal_positive_ratings.csv', index=False)
human_positive_intervals.to_csv(OUT / 'human_temporal_intervals.csv', index=False)
objective_events.to_csv(OUT / 'objective_qtemp_intervals.csv', index=False)

ADJUDICATION = OUT / 'temporal_manual_verification.csv'
if not ADJUDICATION.exists():
    candidates[['logical_recording_id','selected_media_file_name','source_group']].assign(
        reviewer='', review_status='PENDING', disturbance_present='', disturbance_type='', notes=''
    ).to_csv(ADJUDICATION, index=False)

display(pd.DataFrame([{
    'objective_positive_recordings': int(candidates['objective_positive'].sum()),
    'human_positive_unique_recordings': int(candidates['human_positive'].sum()),
    'union_for_review': len(candidates),
    'main_positive_ratings': int(main_ratings['rating'].sum()),
    'reliability_positive_ratings': int(rel_ratings['rating'].sum()),
}]))
display(candidates)

In [ ]:
selector = widgets.Dropdown(
    options=[(f"{r.source_group} | {r.selected_media_file_name}", r.logical_recording_id) for r in candidates.itertuples()],
    description='Recording:', layout=widgets.Layout(width='95%')
)
window_mode = widgets.ToggleButtons(options=['full recording', 'event windows'], value='event windows', description='View:')
reviewer = widgets.Text(description='Reviewer:')
decision = widgets.Dropdown(options=['PENDING', 'PRESENT', 'ABSENT', 'UNCERTAIN'], description='Decision:')
kind = widgets.Text(description='Type:', placeholder='dropout, repetition, splice-like, other')
notes = widgets.Textarea(description='Notes:', layout=widgets.Layout(width='80%'))
save = widgets.Button(description='Save verification', button_style='success')
message = widgets.Output()
viewer = widgets.Output()

def intervals_for(row):
    obj = objective_events.loc[objective_events['logical_recording_id'].eq(row.logical_recording_id)].copy()
    hum = human_positive_intervals.loc[human_positive_intervals['logical_recording_id'].eq(row.logical_recording_id)].copy()
    return obj, hum

def render(*_):
    with viewer:
        clear_output(wait=True)
        row = candidates.loc[candidates['logical_recording_id'].eq(selector.value)].iloc[0]
        audio = decode_native_audio(Path(row['media_path']), ffmpeg='ffmpeg', ffprobe='ffprobe')
        y = audio.native.mean(axis=1, dtype=np.float64)
        fs = audio.sample_rate_native
        obj, hum = intervals_for(row)
        spans = [(float(x.start_sec), float(x.end_sec), 'objective', 'tab:red') for x in obj.itertuples()]
        spans += [(float(x.start_sec), float(x.end_sec), f'human:{x.rater_id}', 'tab:blue') for x in hum.itertuples()]
        duration = len(y) / fs
        if window_mode.value == 'event windows' and spans:
            lo = max(0, min(x[0] for x in spans) - 1.0); hi = min(duration, max(x[1] for x in spans) + 1.0)
        else:
            lo, hi = 0.0, duration
        i0, i1 = int(lo * fs), int(hi * fs)
        view = y[i0:i1]; times = np.arange(i0, i1) / fs
        fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True, constrained_layout=True)
        axes[0].plot(times, view, lw=.45, color='black'); axes[0].set_ylabel('Amplitude')
        nperseg = min(1024, max(64, len(view)))
        f, t, sxx = signal.spectrogram(view, fs=fs, nperseg=nperseg, noverlap=nperseg//2, scaling='spectrum')
        axes[1].pcolormesh(t + lo, f, 10*np.log10(sxx + 1e-12), shading='auto', cmap='magma')
        axes[1].set_ylim(0, min(8000, fs/2)); axes[1].set_ylabel('Frequency (Hz)'); axes[1].set_xlabel('Time (s)')
        used = set()
        for start, end, label, color in spans:
            if end < lo or start > hi: continue
            for ax in axes: ax.axvspan(start, end, color=color, alpha=.25, label=label if label not in used else None)
            used.add(label)
        if used: axes[0].legend(loc='upper right')
        fig.suptitle(f"{row['selected_media_file_name']} | {row['source_group']}")
        plt.show()
        display(pd.DataFrame([row]))
        if len(obj): display(obj[['event_type','event_subtype','start_sec','end_sec','duration_sec','disposition']])
        if len(hum): display(hum[['design','rater_id','subcategory','start_sec','end_sec','duration_sec']])
        print('Full recording:'); display(Audio(y, rate=fs, normalize=False))
        if spans:
            print(f'Review window {lo:.2f}–{hi:.2f} s:'); display(Audio(view, rate=fs, normalize=False))

def save_review(_):
    table = pd.read_csv(ADJUDICATION, keep_default_na=False)
    mask = table['logical_recording_id'].eq(selector.value)
    table.loc[mask, ['reviewer','review_status','disturbance_present','disturbance_type','notes']] = [
        reviewer.value, decision.value, {'PRESENT':'yes','ABSENT':'no','UNCERTAIN':'uncertain'}.get(decision.value,''), kind.value, notes.value
    ]
    table.to_csv(ADJUDICATION, index=False)
    with message:
        clear_output(wait=True); print('Saved:', selector.value, '→', ADJUDICATION)

selector.observe(render, names='value'); window_mode.observe(render, names='value'); save.on_click(save_review)
display(widgets.VBox([selector, window_mode, widgets.HBox([reviewer, decision]), kind, notes, save, message]), viewer)
render()